In [ ]:
# 1. Import Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

# 2. Load the Dataset
print("Loading California Housing dataset...")
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df["Price"] = housing.target

# 3. Feature Engineering (From your Task 2)
print("Performing feature engineering...")
df["RoomsPerHousehold"] = df["AveRooms"] / df["HouseAge"]
df["BedroomsPerRoom"] = df["AveBedrms"] / df["AveRooms"]
df["PopulationPerHousehold"] = df["Population"] / df["AveOccup"]
df["IncomePerRoom"] = df["MedInc"] / df["AveRooms"]

# 4. Data Preparation
print("Splitting and scaling the data...")
X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# TASK 3 IMPLEMENTATION STARTS HERE
# 5. Detect Overfitting (Train vs Test Performance)
print("\n--- Step 5: Overfitting Detection ---")
unconstrained_tree = DecisionTreeRegressor(random_state=42)
unconstrained_tree.fit(X_train_scaled, y_train)

train_pred = unconstrained_tree.predict(X_train_scaled)
test_pred = unconstrained_tree.predict(X_test_scaled)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"Unconstrained Decision Tree Train RMSE: {train_rmse:.4f}")
print(f"Unconstrained Decision Tree Test RMSE:  {test_rmse:.4f}")
print("Interpretation: A large gap between training (often ~0.0) and test RMSE indicates severe overfitting.")

# 6. Cross-Validation (Reliable Evaluation)
print("\n--- Step 6: Cross-Validation ---")
cv_scores = cross_val_score(
    unconstrained_tree, 
    X_train_scaled, 
    y_train, 
    cv=5, 
    scoring='neg_mean_squared_error'
)
cv_rmse_scores = np.sqrt(-cv_scores)

print(f"Cross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.4f} (+/- {cv_rmse_scores.std():.4f})")

# 7. Hyperparameter Tuning Using GridSearchCV
print("\n--- Step 7: Hyperparameter Tuning (GridSearchCV) ---")
param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)
best_params = grid_search.best_params_
print(f"Best Hyperparameters: {best_params}")

# 8. Evaluate Optimized Model
print("\n--- Step 8: Evaluate Optimized Model ---")
best_tree = grid_search.best_estimator_
tuned_test_pred = best_tree.predict(X_test_scaled)

tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_test_pred))
tuned_r2 = r2_score(y_test, tuned_test_pred)

print(f"Tuned Decision Tree Test RMSE: {tuned_rmse:.4f}")
print(f"Tuned Decision Tree Test R2:   {tuned_r2:.4f}")

# 9. Model Comparison Summary Table
print("\n--- Step 9: Model Comparison Summary Table ---")
lr = LinearRegression().fit(X_train_scaled, y_train)
ridge = Ridge(alpha=1.0).fit(X_train_scaled, y_train)

models = {
    "Baseline Linear Regression": lr,
    "Ridge Regression": ridge,
    "Unconstrained Decision Tree": unconstrained_tree,
    "Tuned Decision Tree": best_tree
}

results = []
for name, model in models.items():
    preds = model.predict(X_test_scaled)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results.append({"Model": name, "Test R2": r2, "Test RMSE": rmse})

results_df = pd.DataFrame(results)
print(results_df.sort_values(by="Test R2", ascending=False).to_string(index=False))

# Optional: Save the best model
joblib.dump(best_tree, "best_tuned_tree_model.joblib")
print("\n✅ Task 3 script execution completed successfully!")

Loading California Housing dataset...
Performing feature engineering...
Splitting and scaling the data...

--- Step 5: Overfitting Detection ---
Unconstrained Decision Tree Train RMSE: 0.0000
Unconstrained Decision Tree Test RMSE:  0.7123
Interpretation: A large gap between training (often ~0.0) and test RMSE indicates severe overfitting.

--- Step 6: Cross-Validation ---
Cross-Validation RMSE Scores: [0.73506583 0.73329066 0.72651776 0.72644245 0.74279728]
Mean CV RMSE: 0.7328 (+/- 0.0061)

--- Step 7: Hyperparameter Tuning (GridSearchCV) ---
Best Hyperparameters: {'max_depth': 15, 'min_samples_leaf': 10, 'min_samples_split': 2}

--- Step 8: Evaluate Optimized Model ---
Tuned Decision Tree Test RMSE: 0.6073
Tuned Decision Tree Test R2:   0.7185

--- Step 9: Model Comparison Summary Table ---
                      Model  Test R2  Test RMSE
        Tuned Decision Tree 0.718526   0.607327
 Baseline Linear Regression 0.630574   0.695772
           Ridge Regression 0.630567   0.695778
Unco